# Session-5: CDF-only intra-dataset training

The cross-dataset DFDC+CDF training in session 4 plateaued at AUC 0.685 (balanced).
This notebook trains AND evaluates on Celeb-DF only — the standard intra-dataset
setup used in most published deepfake detection papers. Realistic target:
**AUC 0.92+, accuracy 88-93%**.

Pipeline:
1. Bootstrap (clone, install, CDF symlinks)
2. Build CDF-only manifest with identity-grouped splits
3. Extract train + val + test
4. Train detector with ImageNet-pretrained ResNet50 backbone
5. Evaluate on balanced subset for honest numbers

Total runtime: ~3 h.

## 1. Bootstrap

In [ ]:
import os, shutil, subprocess

os.chdir("/kaggle/working")
if not os.path.exists("code"):
    subprocess.run(["git", "clone", "https://github.com/Parthivsurya/deepfake-detection.git", "code"], check=True)
os.chdir("/kaggle/working/code")
subprocess.run(["git", "pull", "origin", "main"])
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "--no-deps", "facenet-pytorch"])

for d in ["frames", "audio", "manifests", "checkpoints"]:
    os.makedirs(d, exist_ok=True)

# CDF symlinks (doubled-folder quirk from zip + auto-extract)
CDF_SRC = "/kaggle/input/datasets/parthivsuryakb/celeb-df-v2"
assert os.path.exists(CDF_SRC), f"attach celeb-df-v2 first - {CDF_SRC} missing"
os.makedirs("/kaggle/working/cdf_root", exist_ok=True)
for sub in ["Celeb-real", "Celeb-synthesis"]:
    nested = f"{CDF_SRC}/{sub}/{sub}"
    link = f"/kaggle/working/cdf_root/{sub}"
    if not os.path.lexists(link):
        os.symlink(nested, link)

print("ready, pwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-3"])


## 2. Build CDF-only manifest with identity-grouped splits

In [ ]:
%cd /kaggle/working/code
!python scripts/prepare_datasets.py \
    --dataset celebdf \
    --root /kaggle/working/cdf_root \
    --out manifests/

import pandas as pd
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"manifests/{split}.csv")
    print(f"{split}: {len(df)} clips | by label: {df['label'].value_counts().to_dict()}")


## 3. Extract frames (~5,000 CDF clips, ~50 min with GPU MTCNN)

In [ ]:
%cd /kaggle/working/code
import subprocess
for split in ["train", "val", "test"]:
    print(f"\n=== extracting {split} ===")
    ret = subprocess.run([
        "python", "scripts/extract_frames.py",
        "--manifest", f"manifests/{split}.csv",
        "--out_frames", "frames/", "--out_audio", "audio/",
        "--fps", "4", "--max_frames", "32", "--crop_size", "224",
    ])
    print(f"{split} exit: {ret.returncode}")


## 4. Train detector (ResNet50 backbone, 15 epochs)

In [ ]:
%cd /kaggle/working/code
import yaml, os

cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["data"]["manifest_train"] = "manifests/train.extracted.csv"
cfg["data"]["manifest_val"]   = "manifests/val.extracted.csv"
cfg["data"]["batch_size"]     = 8
cfg["data"]["num_workers"]    = 2
cfg["train"]["epochs"]        = 15
cfg["train"]["lr"]            = 1.0e-4
cfg["train"]["weight_decay"]  = 0.05
yaml.safe_dump(cfg, open("configs/session5.yaml", "w"))

if os.path.exists("checkpoints/best.pt"):
    os.remove("checkpoints/best.pt")

!python scripts/train.py --config configs/session5.yaml --device cuda


## 5. Save + push as v4 of dfdc-smoke-artifacts

In [ ]:
%cd /kaggle/working/code
import shutil, json, os
shutil.copy("checkpoints/best.pt", "/kaggle/working/best_session5_cdf_intra.pt")
!ls -lh /kaggle/working/best_session5_cdf_intra.pt

os.makedirs("/kaggle/working/upload5", exist_ok=True)
shutil.copy("/kaggle/working/best_session5_cdf_intra.pt",
            "/kaggle/working/upload5/best_session5_cdf_intra.pt")
with open("/kaggle/working/upload5/dataset-metadata.json", "w") as fp:
    json.dump({"title": "dfdc-smoke-artifacts",
               "id": "parthivsuryakb/dfdc-smoke-artifacts",
               "licenses": [{"name": "CC0-1.0"}]}, fp)
!kaggle datasets version -p /kaggle/working/upload5 -m "v4: CDF intra-dataset, ResNet50 backbone"


## 6. Honest eval — balanced subset of val + threshold sweep + confusion matrix

In [ ]:
%cd /kaggle/working/code
import torch, yaml, numpy as np
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from scripts.train import build_model
from data.datasets import VideoManifest, VideoClipDataset
from torch.utils.data import DataLoader
import pandas as pd

cfg = yaml.safe_load(open("configs/session5.yaml"))
model = build_model(cfg).to("cuda")
ckpt = torch.load("checkpoints/best.pt", map_location="cuda", weights_only=False)
model.load_state_dict(ckpt["model"], strict=False)
model.eval()

val_m = VideoManifest.load("manifests/val.extracted.csv")
df = val_m.df
print(f"val class counts: {df['label'].value_counts().to_dict()}")

# Balanced subset
n_min = df["label"].value_counts().min()
bal = pd.concat([
    df[df["label"]==0].sample(n=n_min, random_state=42),
    df[df["label"]==1].sample(n=n_min, random_state=42),
]).reset_index(drop=True)
bal.to_csv("manifests/val.balanced.csv", index=False)

def eval_on(manifest_path, name):
    m = VideoManifest.load(manifest_path)
    ds = VideoClipDataset(m, num_frames=cfg["data"]["num_frames"],
        frame_size=cfg["data"]["frame_size"],
        audio_sample_rate=cfg["data"]["audio_sample_rate"],
        audio_seconds=cfg["data"]["audio_seconds"])
    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2)
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch["frames"].to("cuda"), batch["audio"].to("cuda"),
                        has_audio=batch["has_audio"].to("cuda"))
            probs.append(F.softmax(out["logits"], dim=-1)[:, 1].cpu().numpy())
            labels.append(batch["label"].numpy())
    p = np.concatenate(probs); y = np.concatenate(labels)
    pred = (p > 0.5).astype(int)
    print(f"\n=== {name} ({len(y)} clips) ===")
    print(f"  acc:  {accuracy_score(y, pred):.4f}")
    print(f"  f1:   {f1_score(y, pred, zero_division=0):.4f}")
    print(f"  auc:  {roc_auc_score(y, p):.4f}")
    cm = confusion_matrix(y, pred)
    print(f"  confusion (true \\ pred):")
    print(f"     true_real | {cm[0,0]:>4} {cm[0,1]:>4}")
    print(f"     true_fake | {cm[1,0]:>4} {cm[1,1]:>4}")
    return p, y

p_full, y_full = eval_on("manifests/val.extracted.csv", "FULL val")
p_bal,  y_bal  = eval_on("manifests/val.balanced.csv",  "BALANCED val")

# Test set too
p_test, y_test = eval_on("manifests/test.extracted.csv", "TEST set")
